# Reto Semanal 3: Pipeline Neuronal para Noticias Falsas

**Nombre: Zhang Tan Rubi**

**Objetivo:** Construir el pipeline completo que transforma noticias del dataset WELFake en tensores listos para una red neuronal, usando los módulos de `src/`.

### Entregables:

| # | Tarea | Descripción |
|---|-------|-------------|
| 1 | Vocabulario | Construye vocabulario de 20,000 palabras SOLO con datos de train |
| 2 | Tokenización | Convierte textos a secuencias de índices (max_len=200) |
| 3 | Dataset | Crea `FakeNewsDataset` para train, val y test |
| 4 | DataLoader | Crea DataLoaders con batch_size=64 |
| 5 | GloVe | Descarga GloVe 6B 50d y crea la matriz de embeddings |
| 6 | nn.Embedding | Inicializa la capa con pesos GloVe |
| 7 | Verificación | Imprime formas de un batch y verifica que sean correctas |
| 8 | Cobertura | ¿Qué porcentaje del vocabulario tiene vector en GloVe? |

### Dimensiones esperadas:
- `batch_texts.shape` = `(64, 200)` — 64 noticias, 200 tokens cada una
- `batch_labels.shape` = `(64,)` — una etiqueta por noticia
- `embedding_matrix.shape` = `(vocab_size, 50)` — un vector 50d por palabra

In [54]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [55]:
import sys
!{sys.executable} -m pip install nltk


In [56]:
import nltk
nltk.download('stopwords')


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [57]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

def remove_stopwords(text):
    words = text.split()
    words = [w for w in words if w.lower() not in stop_words]
    return " ".join(words)


## RETO 3.1 — Cargar datos y construir vocabulario

In [58]:
import sys, os
os.makedirs("/content/drive/MyDrive/ModeladoPredictivo2026/notebooks/src", exist_ok=True)
sys.path.insert(0, "/content/drive/MyDrive/ModeladoPredictivo2026/notebooks")
from src.dataset import build_vocabulary, FakeNewsDataset, create_dataloaders, load_glove, PAD_IDX, UNK_IDX
import pandas as pd
from sklearn.model_selection import train_test_split

In [59]:
# 1. Cargar WELFake
df_reto = pd.read_csv('/content/drive/MyDrive/ModeladoPredictivo2026/data/WELFake_limpio.csv')

# El dataset ya está limpio, solo aseguramos que no haya NaN en texto o etiqueta
df_reto = df_reto.dropna(subset=["text", "label"])
df_reto["label"] = df_reto["label"].astype(int)          # 0 = fake, 1 = real
df_reto["text"] = df_reto["text"].apply(remove_stopwords)

texts  = df_reto["text"].tolist()
labels = df_reto["label"].tolist()

# 2. Split estratificado 80 / 10 / 10
# Primera división: 80% train | 20% temp
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    texts, labels,
    test_size=0.20,
    random_state=42,
    stratify=labels
)

# Segunda división: 50% del 20% restante → 10% val | 10% test
val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels,
    test_size=0.50,
    random_state=42,
    stratify=temp_labels
)

# 3. Construir vocabulario solo con datos de train
word2idx = build_vocabulary(train_texts, max_vocab=20_000)

# 4. Verificación
print(f"Vocabulario construido:")
print(f"  Tamaño total : {len(word2idx):,} palabras")
print(f"  (incluye <PAD>={PAD_IDX} y <UNK>={UNK_IDX})")

# Muestra las 10 palabras más frecuentes
top10 = [(w, i) for w, i in word2idx.items() if 2 <= i <= 11]
top10.sort(key=lambda x: x[1])
print(f"\n  Top 10 palabras más frecuentes:")
for word, idx in top10:
    print(f"    [{idx:>4}]  {word}")


Vocabulario construido:
  Tamaño total : 20,002 palabras
  (incluye <PAD>=0 y <UNK>=1)

  Top 10 palabras más frecuentes:
    [   2]  said
    [   3]  trump
    [   4]  would
    [   5]  mr.
    [   6]  one
    [   7]  people
    [   8]  president
    [   9]  new
    [  10]  also
    [  11]  —


**Observaciones:**
- Se eliminan las *stopwords* antes de construir el vocabulario para que palabras vacías como `the`, `is`, `a` no ocupen lugares entre las 20,000 más frecuentes, dejando espacio a palabras con contenido real.
- El vocabulario se construye **solo con datos de train** para evitar *data leakage*: las palabras de val/test que no aparezcan aquí se mapearán a `<UNK>` durante la inferencia, simulando el comportamiento real en producción.
- Las palabras con índices más bajos (2, 3, 4...) son las **más frecuentes** del corpus de entrenamiento. En un dataset de noticias se espera ver palabras como `said`, `president`, `people`, `government`.
- `<PAD>` (índice 0) y `<UNK>` (índice 1) se reservan siempre como primeras entradas.

## RETO 3.2 — Crear DataLoaders

In [60]:
import torch

# 1. Empaquetar los splits en tuplas (textos, etiquetas)
train_data = (train_texts, train_labels)
val_data   = (val_texts,   val_labels)
test_data  = (test_texts,  test_labels)

# 2. Crear los tres DataLoaders
train_loader, val_loader, test_loader = create_dataloaders(
    train_data=train_data,
    val_data=val_data,
    test_data=test_data,
    word2idx=word2idx,
    batch_size=64,
    max_len=200,
)

# 3. Inspeccionar un batch
batch_texts, batch_labels = next(iter(train_loader))

print("\nVerificación de shapes:")
print(f"  batch_texts.shape  = {tuple(batch_texts.shape)}")
print(f"  batch_labels.shape = {tuple(batch_labels.shape)}")
print(f"  batch_texts.dtype  = {batch_texts.dtype}")
print(f"  batch_labels.dtype = {batch_labels.dtype}")

# 4. Verificaciones de contenido
# ¿Los índices están dentro del rango válido del vocabulario?
assert batch_texts.max().item() < len(word2idx), "Índice fuera del vocabulario"
assert batch_texts.min().item() >= 0,            "Índice negativo"

unique_labels = batch_labels.unique().tolist()
assert all(l in (0.0, 1.0) for l in unique_labels), "Etiquetas inesperadas"

print("\nContenido del batch:")
print(f"  Índice mínimo en batch : {batch_texts.min().item()}")
print(f"  Índice máximo en batch : {batch_texts.max().item()}")
print(f"  Etiquetas únicas       : {[int(l) for l in unique_labels]}")
print(f"  Noticias fake    (0)   : {(batch_labels == 0).sum().item()}")
print(f"  Noticias reales  (1)   : {(batch_labels == 1).sum().item()}")

# 5. Revisar padding en una muestra
# Cuenta cuántos tokens son PAD (índice 0) en la primera noticia del batch
primera_noticia = batch_texts[0]
n_pad    = (primera_noticia == 0).sum().item()
n_tokens = 200 - n_pad
print(f"\nPrimera noticia del batch:")
print(f"  Tokens reales : {n_tokens}")
print(f"  Padding (PAD) : {n_pad}")
print(f"  Primeros 10 índices : {primera_noticia[:10].tolist()}")


DataLoaders creados:
  Entrenamiento: 50,488 muestras -> 789 batches
  Validacion:    6,311 muestras -> 99 batches
  Prueba:        6,311 muestras -> 99 batches
  Batch size: 64 | Max len: 200

Verificación de shapes:
  batch_texts.shape  = (64, 200)
  batch_labels.shape = (64,)
  batch_texts.dtype  = torch.int64
  batch_labels.dtype = torch.float32

Contenido del batch:
  Índice mínimo en batch : 0
  Índice máximo en batch : 19982
  Etiquetas únicas       : [0, 1]
  Noticias fake    (0)   : 43
  Noticias reales  (1)   : 21

Primera noticia del batch:
  Tokens reales : 200
  Padding (PAD) : 0
  Primeros 10 índices : [65, 61, 26, 118, 27, 134, 262, 4, 1335, 891]


**Observaciones:**
- La forma `(64, 200)` confirma que el `DataLoader` agrupa correctamente 64 noticias, cada una representada como una secuencia de 200 índices enteros.
- `batch_texts.dtype = torch.int64` es el tipo correcto para índices de vocabulario que se pasarán a `nn.Embedding`.
- `batch_labels.dtype = torch.float32` es necesario porque la función de pérdida `BCEWithLogitsLoss` opera con flotantes.
- Los `assert` verifican que no haya índices fuera del vocabulario (lo que causaría un error en la capa de embedding) ni etiquetas con valores inesperados.
- La proporción de tokens reales vs. padding en la primera noticia refleja la longitud real de ese artículo después de eliminar stopwords. Al haber aplicado remove_stopwords() antes de tokenizar, los textos son más cortos que en el dataset original, por lo que es esperable ver una cantidad considerable de padding incluso con max_len=200. Esto no representa un problema: la capa de embedding tiene padding_idx=0, lo que indica a PyTorch que no propague gradientes a través de las posiciones de padding durante el entrenamiento.

## RETO 3.3 — Cargar GloVe y crear nn.Embedding

In [61]:
import torch.nn as nn
import numpy as np
from pathlib import Path

# 1. Cargar GloVe
GLOVE_PATH_R = Path('/content/drive/MyDrive/ModeladoPredictivo2026/data/glove.6B.50d.txt')
EMBED_DIM_R  = 50

if GLOVE_PATH_R.exists():
    embedding_matrix = load_glove(
        glove_path=str(GLOVE_PATH_R),
        word2idx=word2idx,
        embed_dim=EMBED_DIM_R,
    )
else:
    print("GloVe no encontrado — usando inicialización aleatoria temporal")
    embedding_matrix = np.random.normal(scale=0.6, size=(len(word2idx), EMBED_DIM_R))
    embedding_matrix[PAD_IDX] = np.zeros(EMBED_DIM_R)

print(f"\nForma de la matriz : {embedding_matrix.shape}")

# 2. Crear nn.Embedding e inicializar con pesos GloVe
vocab_size, embed_dim = embedding_matrix.shape

embedding_layer = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=embed_dim,
    padding_idx=PAD_IDX,
)

# Reemplazar los pesos aleatorios por los vectores GloVe
embedding_layer.weight = nn.Parameter(
    torch.tensor(embedding_matrix, dtype=torch.float32)
)

print(f"nn.Embedding creado : ({vocab_size:,} palabras × {embed_dim} dims)")

# 3. Verificar que el vector de PAD sea exactamente ceros
pad_vector = embedding_layer.weight[PAD_IDX].detach()
es_cero    = torch.all(pad_vector == 0).item()

print(f"\nVector PAD (índice {PAD_IDX}):")
print(f"  ¿Es todo ceros? {'Sí' if es_cero else 'No'}")
print(f"  Norma L2       : {pad_vector.norm().item():.6f}")

# 4. Cobertura real del vocabulario
palabras_con_glove = 0
if GLOVE_PATH_R.exists():
    with open(str(GLOVE_PATH_R), "r", encoding="utf-8") as f:
        glove_vocab = {line.split()[0] for line in f}

    for word in word2idx:
        if word in glove_vocab:
            palabras_con_glove += 1

    cobertura  = palabras_con_glove / vocab_size * 100
    sin_vector = vocab_size - palabras_con_glove

    print(f"\nCobertura GloVe sobre el vocabulario:")
    print(f"  Con vector GloVe    : {palabras_con_glove:>6,}  ({cobertura:.1f}%)")
    print(f"  Sin vector (random) : {sin_vector:>6,}  ({100-cobertura:.1f}%)")
    print(f"  Total vocabulario   : {vocab_size:>6,}")

# 5. Prueba rápida: pasar un batch real por la capa
batch_texts_test, _ = next(iter(train_loader))
with torch.no_grad():
    embedded = embedding_layer(batch_texts_test)

print(f"\nPrueba con batch real:")
print(f"  Input  shape : {tuple(batch_texts_test.shape)}")
print(f"  Output shape : {tuple(embedded.shape)}")
assert embedded.shape == (64, 200, 50), "Shape inesperado"


GloVe cargado: 13,585 / 20,002 palabras encontradas (67.9% de cobertura)

Forma de la matriz : (20002, 50)
nn.Embedding creado : (20,002 palabras × 50 dims)

Vector PAD (índice 0):
  ¿Es todo ceros? Sí
  Norma L2       : 0.000000

Cobertura GloVe sobre el vocabulario:
  Con vector GloVe    : 13,585  (67.9%)
  Sin vector (random) :  6,417  (32.1%)
  Total vocabulario   : 20,002

Prueba con batch real:
  Input  shape : (64, 200)
  Output shape : (64, 200, 50)


**Observaciones:**
- La matriz de embeddings tiene forma `(vocab_size, 50)`: cada fila `i` contiene el vector de 50 dimensiones de la palabra con `word2idx[palabra] == i`.
- El vector de `<PAD>` debe ser **exactamente ceros** (norma L2 = 0.0). Esto evita que las posiciones de padding aporten información a la red; además, `padding_idx=0` hace que PyTorch no actualice ese vector durante el entrenamiento.
- La prueba final confirma que el pipeline completo funciona de punta a punta: un tensor `(64, 200)` de índices entra a `nn.Embedding` y produce un tensor `(64, 200, 50)` de vectores densos, listo para una capa LSTM o CNN.

In [62]:
from collections import Counter

GLOVE_PATH = Path('/content/drive/MyDrive/ModeladoPredictivo2026/data/glove.6B.50d.txt')

# Reconstruir glove_words
glove_words = set()
if GLOVE_PATH.exists():
    with open(str(GLOVE_PATH), "r", encoding="utf-8") as f:
        for line in f:
            word = line.split()[0]
            if word in word2idx:
                glove_words.add(word)

# Frecuencias en train
counter_train = Counter()
for text in train_texts:
    if isinstance(text, str):
        counter_train.update(text.lower().split())

palabras_sin_glove = [
    w for w in word2idx
    if w not in glove_words and w not in {"<PAD>", "<UNK>"}
]

palabras_sin_glove.sort(key=lambda w: counter_train.get(w, 0), reverse=True)

print(f"Estadísticas de cobertura GloVe\n")
print(f"Tamaño del vocabulario:             {len(word2idx):,}")
print(f"Palabras CON vector GloVe:          {len(glove_words):,} ({len(glove_words)/len(word2idx)*100:.1f}%)")
print(f"Palabras SIN vector GloVe (random): {len(palabras_sin_glove):,} ({len(palabras_sin_glove)/len(word2idx)*100:.1f}%)")

print(f"\nTop 20 palabras sin vector GloVe (por frecuencia en train):")
for w in palabras_sin_glove[:20]:
    print(f"  {counter_train[w]:>6,}x  '{w}'")


Estadísticas de cobertura GloVe

Tamaño del vocabulario:             20,002
Palabras CON vector GloVe:          13,585 (67.9%)
Palabras SIN vector GloVe (random): 6,415 (32.1%)

Top 20 palabras sin vector GloVe (por frecuencia en train):
  19,114x  'trump’s'
  16,939x  '(reuters)'
  15,339x  'it’s'
  15,246x  'said,'
  14,185x  '“the'
  13,146x  '“i'
  11,389x  'don’t'
  10,623x  'it.'
   9,740x  'trump,'
   8,229x  'however,'
   7,909x  '“we'
   6,266x  'that’s'
   6,201x  'year,'
   5,961x  'it,'
   5,870x  'trump.'
   5,635x  'that,'
   5,463x  'clinton’s'
   5,333x  'them.'
   4,963x  'time,'
   4,938x  'states,'


**Observaciones:**
- Una cobertura del 60–80% es típica y aceptable. La causa principal de las palabras sin cobertura en este dataset es la puntuación no separada: tokens como said,, trump,, it's, don't o "the son palabras que GloVe sí conoce (said, trump, it, the), pero al conservar la coma, el apóstrofo o la comilla tipográfica pegados, el token no coincide con ninguna entrada del vocabulario GloVe.
Si en el Notebook 02 se hubiera aplicado una separación de signos de puntuación, la cobertura subiría considerablemente.
Además, también aparecen nombres propios específicos del dominio de noticias como (reuters) y algunas contracciones en inglés. Las palabras más frecuentes con contenido semántico real casi siempre están cubiertas, las que no tienen vector GloVe reciben inicialización aleatoria que el modelo ajustará durante el entrenamiento.

- Ordenar por frecuencia en train es útil: si alguna palabra muy frecuente no tiene vector GloVe, el modelo deberá aprenderla desde cero durante el entrenamiento, lo que puede requerir más épocas para converger.



---
## Extra: max_len=512

In [63]:
# Experimento: mismo pipeline con max_len=512 (percentil ~75 del dataset)
# El EDA mostró que con max_len=200 se trunca el 79.2% de los textos
# max_len=512 cubre mejor la distribución sin desperdiciar demasiada memoria

from src.dataset import text_to_indices

MAX_LEN_512 = 512

train_loader_512, val_loader_512, test_loader_512 = create_dataloaders(
    train_data=(train_texts, train_labels),
    val_data=(val_texts, val_labels),
    test_data=(test_texts, test_labels),
    word2idx=word2idx,
    batch_size=64,
    max_len=MAX_LEN_512,
)

batch_texts_512, batch_labels_512 = next(iter(train_loader_512))

print(f"Shapes del batch:")
print(f"  max_len=200 → {tuple(batch_texts.shape)}")
print(f"  max_len=512 → {tuple(batch_texts_512.shape)}")

# Comparación de la misma noticia con los dos max_len
texto_ejemplo = train_texts[0]
n_palabras = len(texto_ejemplo.split())

indices_200 = text_to_indices(texto_ejemplo, word2idx, max_len=200)
indices_512 = text_to_indices(texto_ejemplo, word2idx, max_len=512)

n_pad_200 = indices_200.count(0)
n_pad_512 = indices_512.count(0)

print(f"\nComparación de padding (misma noticia, {n_palabras} palabras reales):")
print(f"  max_len=200 → tokens reales: {200 - n_pad_200:>3}, padding: {n_pad_200:>3} ({n_pad_200/200*100:.1f}%)")
print(f"  max_len=512 → tokens reales: {512 - n_pad_512:>3}, padding: {n_pad_512:>3} ({n_pad_512/512*100:.1f}%)")

# Verificar embedding
with torch.no_grad():
    embedded_512 = embedding_layer(batch_texts_512)

print(f"\nOutput del embedding:")
print(f"  max_len=200 → {tuple(embedding_layer(batch_texts).shape)}")
print(f"  max_len=512 → {tuple(embedded_512.shape)}")

DataLoaders creados:
  Entrenamiento: 50,488 muestras -> 789 batches
  Validacion:    6,311 muestras -> 99 batches
  Prueba:        6,311 muestras -> 99 batches
  Batch size: 64 | Max len: 512
Shapes del batch:
  max_len=200 → (64, 200)
  max_len=512 → (64, 512)

Comparación de padding (misma noticia, 148 palabras reales):
  max_len=200 → tokens reales: 148, padding:  52 (26.0%)
  max_len=512 → tokens reales: 148, padding: 364 (71.1%)

Output del embedding:
  max_len=200 → (64, 200, 50)
  max_len=512 → (64, 512, 50)


**Observación:**

Con `max_len=512` el tensor de salida del embedding es `(64, 512, 50)` en lugar de `(64, 200, 50)`.

Para una noticia de 148 palabras reales, `max_len=200` genera 52 tokens de padding (26%) mientras que `max_len=512` genera 364 (71%), padding innecesario que consume memoria sin aportar información.

Sin embargo, `max_len=512` es más beneficioso para noticias largas: dado que el 79.2% del dataset supera las 200 palabras, con ese valor se pierde información por truncado en la mayoría de los textos.